In [1]:
#============================================================
# Celda 1 — Setup y rutas
#============================================================
import pandas as pd
import numpy as np
from pathlib import Path
import os
os.chdir("/workspaces/TFG_Spain-s_Migratory_Flow")

SILVER_DIR = Path("output/conectividad/02-silver")
GOLD_DIR   = Path("output/conectividad/03-gold")
GOLD_DIR.mkdir(parents=True, exist_ok=True)

assert (SILVER_DIR / "cobertura_provincia_anio.parquet").exists(), "❌ Ejecuta primero el 02"
print("✅ Silver encontrado")

✅ Silver encontrado


In [2]:
#============================================================
# Celda 2 — Leer silver, convertir proporción → porcentaje
#============================================================
df = pd.read_parquet(SILVER_DIR / "cobertura_provincia_anio.parquet")

# Los valores están en proporción 0-1 → convertir a % 0-100
df["cob_100mbps_pct"] = (df["cob_100mbps"] * 100).round(4)
df = df.drop(columns=["cob_100mbps"])

print(f"Shape: {df.shape}")
print(df.head(5).to_string())
print(f"\nEstadísticas:")
print(df["cob_100mbps_pct"].describe().round(2))

Shape: (624, 4)
   Comunidad Autónoma Provincia   año  cob_100mbps_pct
0  Castilla-La Mancha  Albacete  2013          52.5929
1  Castilla-La Mancha  Albacete  2014          52.9683
2  Castilla-La Mancha  Albacete  2015          53.7551
3  Castilla-La Mancha  Albacete  2016          53.6744
4  Castilla-La Mancha  Albacete  2017          62.1609

Estadísticas:
count    624.00
mean      71.20
std       21.30
min        1.52
25%       56.42
50%       75.07
75%       90.31
max      100.00
Name: cob_100mbps_pct, dtype: float64


In [3]:
#============================================================
# Celda 3 — Variación interanual (delta YoY)
#============================================================
df = df.sort_values(["Provincia", "año"]).reset_index(drop=True)

df["delta_yoy"] = (
    df.groupby("Provincia")["cob_100mbps_pct"]
    .diff()
    .round(4)
)

print("Muestra con delta_yoy:")
print(df[df["Provincia"] == "Albacete"].to_string())
print(f"\nNulos en delta_yoy: {df['delta_yoy'].isna().sum()} (esperado: 52, uno por provincia)")

Muestra con delta_yoy:
    Comunidad Autónoma Provincia   año  cob_100mbps_pct  delta_yoy
0   Castilla-La Mancha  Albacete  2013          52.5929        NaN
1   Castilla-La Mancha  Albacete  2014          52.9683     0.3754
2   Castilla-La Mancha  Albacete  2015          53.7551     0.7868
3   Castilla-La Mancha  Albacete  2016          53.6744    -0.0807
4   Castilla-La Mancha  Albacete  2017          62.1609     8.4865
5   Castilla-La Mancha  Albacete  2018          74.6987    12.5378
6   Castilla-La Mancha  Albacete  2019          88.5003    13.8016
7   Castilla-La Mancha  Albacete  2020          92.0779     3.5776
8   Castilla-La Mancha  Albacete  2021          87.5416    -4.5363
9   Castilla-La Mancha  Albacete  2022          90.0656     2.5240
10  Castilla-La Mancha  Albacete  2023          92.5153     2.4497
11  Castilla-La Mancha  Albacete  2024          92.9757     0.4604

Nulos en delta_yoy: 52 (esperado: 52, uno por provincia)


In [4]:
#============================================================
# Celda 4 — Ranking cobertura dentro de cada año
#============================================================
df["ranking_anio"] = (
    df.groupby("año")["cob_100mbps_pct"]
    .rank(ascending=False, method="min")
    .astype(int)
)

print("Top 5 provincias en 2024:")
top2024 = df[df["año"] == 2024].sort_values("ranking_anio").head(5)
print(top2024[["Provincia", "cob_100mbps_pct", "ranking_anio"]].to_string(index=False))

print("\nBottom 5 provincias en 2024:")
bot2024 = df[df["año"] == 2024].sort_values("ranking_anio", ascending=False).head(5)
print(bot2024[["Provincia", "cob_100mbps_pct", "ranking_anio"]].to_string(index=False))

Top 5 provincias en 2024:
Provincia  cob_100mbps_pct  ranking_anio
   Madrid          99.0966             1
Barcelona          98.8320             2
  Melilla          98.2784             3
  Bizkaia          97.6577             4
Cantabria          97.5371             5

Bottom 5 provincias en 2024:
Provincia  cob_100mbps_pct  ranking_anio
    Soria          68.8000            52
  Ourense          70.9643            51
   Teruel          75.1114            50
     Lugo          76.1270            49
    Ávila          79.5464            48


In [5]:
#============================================================
# Celda 5 — Renombrar a snake_case estándar del proyecto
#============================================================
df = df.rename(columns={
    "Comunidad Autónoma": "ccaa",
    "Provincia":          "provincia",
    "año":                "anyo"
})

print(f"Columnas finales: {list(df.columns)}")
print(f"Shape: {df.shape}")
print(df.head(3).to_string())

Columnas finales: ['ccaa', 'provincia', 'anyo', 'cob_100mbps_pct', 'delta_yoy', 'ranking_anio']
Shape: (624, 6)
                 ccaa provincia  anyo  cob_100mbps_pct  delta_yoy  ranking_anio
0  Castilla-La Mancha  Albacete  2013          52.5929        NaN            20
1  Castilla-La Mancha  Albacete  2014          52.9683     0.3754            22
2  Castilla-La Mancha  Albacete  2015          53.7551     0.7868            26


In [6]:
#============================================================
# Celda 6 — Guardar Gold parquet + CSV
#============================================================
df.to_parquet(GOLD_DIR / "gold_conectividad_provincia_anio.parquet", index=False)
df.to_csv(GOLD_DIR    / "gold_conectividad_provincia_anio.csv",     index=False)

print(f"✅ Guardado en {GOLD_DIR}:")
print(f"   gold_conectividad_provincia_anio.parquet")
print(f"   gold_conectividad_provincia_anio.csv")
print(f"\nShape: {df.shape}")
print(f"Años: {sorted(df['anyo'].unique())}")
print(f"Provincias: {df['provincia'].nunique()}")
print(f"\nDtypes:\n{df.dtypes}")

✅ Guardado en output/conectividad/03-gold:
   gold_conectividad_provincia_anio.parquet
   gold_conectividad_provincia_anio.csv

Shape: (624, 6)
Años: [np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]
Provincias: 52

Dtypes:
ccaa                   str
provincia              str
anyo                 int64
cob_100mbps_pct    float64
delta_yoy          float64
ranking_anio         int64
dtype: object
